# Educative Exploration: Active Directory Graph Pipeline

### Pipeline Architecture
The pipeline is divided into four main phases:
1. **Generation & Orchestration** (`generate.sh` & `generate_configs.py`)
2. **Graph Extraction & Post-Processing** (`process_graph.py`)
3. **Mathematical Risk Simulation** (`random_best_alloc.py`)
4. **Graph Neural Network Learning** (`test_graph.py`)

---
## Phase 1: Generation & Orchestration
### `generate_configs.py` & `generate.sh`

The `generate_configs.py` script ensures variance by randomizing domain structures and vulnerabilities (ACL misconfigurations, RDP access, etc.). 

The bash script (`generate.sh`) does the full adsimulator interface and write the jsonl output
1. It generates the config.
2. It restarts Neo4j (to wipe old graphs).
3. It feeds the config into `adsimulator`.
4. It uses Neo4j's `APOC` plugin to export the graph as a `.json` line file, in fact it's a jsonl structure.

Let's look at how the probability normalization works in the config generator:

In [ ]:
import random
import json

def explain_config_generation():
    # We randomize ACL (Access Control List) vulnerabilities
    acls = {
        "GenericAll": random.randint(5, 15),
        "AddMember": random.randint(30, 70)  # Make adding members more common
    }
    
    total = sum(acls.values())
    print(f"Raw randomized sum: {total} (Needs to be exactly 100 for adsimulator)")
    
    # Normalization: ensuring probabilities sum to exactly 100%
    for k in acls:
        acls[k] = int((acls[k] / total) * 100)
        
    diff = 100 - sum(acls.values())
    acls["AddMember"] += diff # Pad any rounding errors into the most common ACL
    
    print(f"Normalized probabilities: {acls}")
    print(f"Final sum: {sum(acls.values())}")

explain_config_generation()

---
## Phase 2: Graph Extraction & Post-Processing
### `process_graph.py`

A raw AD graph contains thousands of nodes, but an attacker only cares about the paths from **Compromised Users** to **High-Value Targets** (like Domain Controllers).


The `extract_attack_subgraph` function is what ensure to extract that subgraph. It uses a **Bidirectional BFS (Breadth-First Search)** to mathematically guarantee the extraction of *only* relevant nodes. It create the surface of attack

1. **Forward BFS**: Calculates distance from sources to all nodes.
2. **Backward BFS**: Calculates distance from targets backward to all nodes.
3. **Intersection**: If `$ Distance_{source} + Distance_{target} <= MaxHops $`, the node is on a valid attack path.

In [ ]:
import networkx as nx

def demonstrate_bidirectional_bfs():
    # Create a dummy graph
    G = nx.DiGraph()
    G.add_edges_from([('Source', 'A'), ('A', 'Target'), ('Source', 'B'), ('B', 'DeadEnd')])
    
    sources, targets = ['Source'], ['Target']
    max_hops = 2
    
    # 1. Forward Distances
    dist_from_src = nx.single_source_shortest_path_length(G, 'Source', cutoff=max_hops)
    print(f"Distances from Source: {dist_from_src}")
    
    # 2. Backward Distances (reverse graph)
    G_rev = G.reverse(copy=False)
    dist_to_tgt = nx.single_source_shortest_path_length(G_rev, 'Target', cutoff=max_hops)
    print(f"Distances to Target: {dist_to_tgt}")
    
    # 3. Intersection Logic
    valid_nodes = set()
    for node, d_S in dist_from_src.items():
        if node in dist_to_tgt:
            if d_S + dist_to_tgt[node] <= max_hops:
                valid_nodes.add(node)
                
    print(f"\nNodes mathematically proven to be on an attack path: {valid_nodes}")
    print("Notice 'DeadEnd' and 'B' are excluded!")

demonstrate_bidirectional_bfs()